# Fine-Tune with clDice Loss

Resume from the U-Net++ + EfficientNet-B4 baseline and fine-tune with
a **clDice + Dice** compound loss to target thin-road failures
surfaced in notebook `08_metric_analysis.ipynb`.

**Setup:**
- Branch: `metric-analysis` (do **not** run on `main`)
- Colab GPU: A100 or T4 both work (batch_size halved for clDice compute cost)
- Checkpoint: resume from the best baseline `.pth` (0.7016 IoU) on Drive
- Config: `configs/finetune_cldice.yaml`
- Expected: ~1h on A100, 15 epochs max

The config uses `iters=5` for soft skeletonization — each iteration
peels ~2 px of width, so the centerline emerges for every road ≤10 px
wide (the "narrow" bucket from the width analysis).


---
## 1 · Environment

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
BRANCH = "metric-analysis"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kagglehub", "wandb", "python-dotenv"], check=True)
    print("Environment ready.")
else:
    REPO_DIR = None


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

branch = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git branch:   {branch}")
assert branch == BRANCH, f"Expected branch {BRANCH}, got {branch!r}"


In [ ]:
# ====== CREDENTIALS ======
os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""
os.environ["WANDB_API_KEY"] = ""  # optional; leave empty to disable W&B
os.environ["WANDB_PROJECT"] = "road-segmentation"

assert os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_KEY"], "Kaggle creds required"


In [ ]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    device = torch.device("cpu")
    print("No GPU detected — training will be impractical.")
print(f"Device: {device}")


---
## 2 · Dataset + Drive

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR

train_dir = DEEPGLOBE_DATASET_DIR / "train"
if not (train_dir.exists() and any(train_dir.glob("*_sat.*"))):
    from road_segmentation.data.download import download_dataset
    download_dataset()
print(f"Dataset: {DEEPGLOBE_DATASET_DIR}")


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/road_segmentation")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR = str(DRIVE_DIR / "checkpoints")
    LOG_DIR = str(DRIVE_DIR / "logs")
else:
    CHECKPOINT_DIR = "checkpoints"
    LOG_DIR = "logs"
print(f"Checkpoints: {CHECKPOINT_DIR}")


---
## 3 · Load config + point at the baseline checkpoint

Set `BASELINE_CHECKPOINT` to the best `.pth` from the original training
run (the one that produced 0.7016 IoU — either the base B4-1024 or
the boundary-fine-tuned one, whichever is your strongest).


In [ ]:
from road_segmentation.config import load_config

config = load_config(PROJECT_ROOT / "configs" / "finetune_cldice.yaml")

# ====== SET BASELINE CHECKPOINT ======
BASELINE_CHECKPOINT = ""  # <-- path to your best.pth on Drive
if IN_COLAB and not BASELINE_CHECKPOINT:
    candidates = sorted(Path(CHECKPOINT_DIR).rglob("best.pth"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        BASELINE_CHECKPOINT = str(candidates[0])
        print(f"Auto-picked latest best.pth: {BASELINE_CHECKPOINT}")

assert BASELINE_CHECKPOINT and Path(BASELINE_CHECKPOINT).exists(), \
    f"Set BASELINE_CHECKPOINT to your best.pth (got {BASELINE_CHECKPOINT!r})"

config.checkpoint.resume_from = BASELINE_CHECKPOINT
config.checkpoint.save_dir = CHECKPOINT_DIR
config.logging.log_dir = LOG_DIR
config.data.num_workers = 2 if IN_COLAB else 4
config.training.mixed_precision = torch.cuda.is_available()
config.logging.wandb = bool(os.environ.get("WANDB_API_KEY"))

print(f"Resume from: {config.checkpoint.resume_from}")
print(f"Save to:     {CHECKPOINT_DIR}/{config.logging.experiment_name}")
print(f"Loss:        {config.loss.type}  (iters={config.loss.params['iters']})")
print(f"Epochs:      {config.training.epochs}")
print(f"Batch size:  {config.data.batch_size}  (×{config.training.grad_accumulation_steps} accum)")
print(f"LR:          {config.optimizer.lr}")


---
## 4 · Resume-safe: auto-pick last clDice checkpoint

If the Colab runtime disconnects during training, re-running this
notebook will transparently resume from `last.pth` in the clDice
experiment directory (not the baseline checkpoint).


In [ ]:
cldice_ckpt_dir = Path(CHECKPOINT_DIR) / config.logging.experiment_name
last_ckpt = cldice_ckpt_dir / "last.pth"
if last_ckpt.exists():
    config.checkpoint.resume_from = str(last_ckpt)
    print(f"Resuming clDice training from: {last_ckpt}")
else:
    print(f"Starting clDice fine-tune from baseline: {BASELINE_CHECKPOINT}")


---
## 5 · Build data + model + loss

In [ ]:
import random, numpy as np

from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_train_transform, get_val_transform
from road_segmentation.data.dataset import create_dataloaders
from road_segmentation.models.factory import create_model
from road_segmentation.training.losses import create_loss
from road_segmentation.training.trainer import Trainer

random.seed(config.seed); np.random.seed(config.seed); torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
train_pairs, val_pairs = split_pairs(
    pairs, val_ratio=config.data.val_split_ratio, seed=config.data.val_split_seed,
)

train_transform = get_train_transform(
    config.data.image_size, config.augmentations.train,
    config.normalization.mean, config.normalization.std,
)
val_transform = get_val_transform(
    config.data.image_size, config.normalization.mean, config.normalization.std,
)

train_loader, val_loader = create_dataloaders(
    train_pairs, val_pairs, train_transform, val_transform,
    batch_size=config.data.batch_size,
    num_workers=config.data.num_workers,
    pin_memory=device.type == "cuda",
)
print(f"Data: {len(train_pairs)} train, {len(val_pairs)} val")

model = create_model(
    arch=config.model.arch,
    encoder_name=config.model.encoder_name,
    encoder_weights=config.model.encoder_weights,
    in_channels=config.model.in_channels,
    classes=config.model.classes,
)

loss_fn = create_loss(config.loss.type, config.loss.params)
print(f"Loss: {type(loss_fn).__name__} — {config.loss.type}")


---
## 6 · Quick clDice sanity check

Verify `soft_skeletonize` centerlines a 10-px wide stripe (that's the
edge of the narrow bucket). The output should concentrate around the
middle row of the stripe.


In [ ]:
from road_segmentation.training.losses import soft_skeletonize, CLDiceLoss
import matplotlib.pyplot as plt

stripe = torch.zeros(1, 1, 64, 64)
stripe[0, 0, 27:37, :] = 1.0  # 10px-wide horizontal stripe
skel = soft_skeletonize(stripe, iters=5)[0, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(stripe[0, 0], cmap="gray"); axes[0].set_title("Input (10 px wide)")
axes[1].imshow(skel, cmap="hot"); axes[1].set_title(f"Soft skeleton (iters=5)\nrow mass peak at y={skel.sum(1).argmax()}")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

# clDice(gt, gt) should be ~0
cld = CLDiceLoss(iters=5, from_logits=False)
gt_rand = (torch.rand(1, 1, 64, 64) > 0.9).float()
print(f"clDice(gt, gt) = {cld(gt_rand, gt_rand).item():.4f}  (should be near 0)")


---
## 7 · Train

Runs 15 epochs max, early stopping on `val_loss` (see config — we
deliberately don't use `val_iou` because aggregate IoU may plateau
while clDice actually improves thin-road recall).


In [ ]:
trainer = Trainer(
    config=config,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)

trainer.train()


---
## 8 · Locate the best clDice checkpoint

Next step: re-run `notebooks/10_metric_analysis_cldice.ipynb` with
this checkpoint path to compare stratified metrics against the baseline.


In [ ]:
best_ckpt = Path(CHECKPOINT_DIR) / config.logging.experiment_name / "best.pth"
print(f"Best clDice checkpoint: {best_ckpt}")
print(f"Exists: {best_ckpt.exists()}")
if best_ckpt.exists():
    print(f"Size: {best_ckpt.stat().st_size / 1e6:.1f} MB")
